# Decision tree met Gini 

Je kunt een library gebruiken om Decision tree te trainen. Hieronder geven we een voorbeeld voor de MNIST database.

Let er op dat de enige features die hier worden gebruikt pixelwaardes op locaties zijn. Dus:

- enkel rauwe pixel intensiteiten
- 1 pixel = 1 numerical feature
- Er is geen feature die gaat over ruimtelijk begrip, filtering, of anderen (eg symmetrie in verticale as, aantal gaten etc)

Importeer de libaries, laad de mnist db en spilt test en training set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report


mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist['data']
y = mnist['target'] 
y = y.astype(int)       

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Trainen van een decision tree is nu heel simpel. We gebruiken ***max_depth = 20*** om er voor te zorgen dat de decision tree niet overtraint. 

Wat is overtrainen hier en waarom denk je dat het nodig is?

Overtrainen betekent dat de boom zo diep groeit dat hij de trainingsdata uit zijn hoofd leert in plaats van algemene patronen. Zonder max_depth groeit de boom totdat elk blad één trainingsplaatje bevat: 100% nauwkeurig op train, maar slecht op nieuwe plaatjes. max_depth=20 voorkomt dat.

In [ ]:
dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=20,    
    random_state=42
)

dt.fit(X_train, y_train)

Doe nu de voorspellingen en laat de precisie zien.

In [ ]:
y_pred = dt.predict(X_test)


acc = accuracy_score(y_test, y_pred)
print("Decision Tree Accuracy on MNIST:", acc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

- Zoek uit waar de waardes in de classification report voor staan of je eigen eerder gemaakte keuzes toe te lichten.

- Is de waarde hoger dan wat jij in je eerdere Decision tree had? Waarom denk je?

**precision**: van alle keren dat het model dit cijfer voorspelde, hoe vaak klopte dat.  
**recall**: van alle echte plaatjes van dit cijfer, hoeveel werden correct herkend.  
**f1-score**: combinatie van precision en recall.  
**support**: aantal testplaatjes voor dat cijfer.

Ja, de accuracy is veel hoger dan de ~39,6% van de P3 boom. Sklearn gebruikt alle 784 pixels als features in plaats van 11 handgemaakte features, heeft meer trainingsdata (56.000 vs 5.000) en vindt betere splitsingen.

Je kunt de boom ook met een library laten zien. Let er op dat je een limiet instelt op hoeveel lagen je toont (gebruik **max_depth=klein_getal**)

In [ ]:
from sklearn import tree
plt.figure(figsize=(20,10))
tree.plot_tree(dt, max_depth=1, filled=True)
plt.show()

Laten we nu kijken of je betere resultaten kunt krijgen uit de features die je zelf eerder hebt bedacht.

In [ ]:
DREMPEL = 128

def procent_donkere_pixels(img):
    return np.sum(img > DREMPEL) / img.size

def links_vs_rechts(img):
    links  = np.mean(img[:, :14].astype(float))
    rechts = np.mean(img[:, 14:].astype(float))
    return links - rechts

def boven_vs_onder(img):
    boven = np.mean(img[:14].astype(float))
    onder = np.mean(img[14:].astype(float))
    return boven - onder

def symmetrie_horizontaal(img):
    spiegel = img[:, ::-1]
    return np.mean(np.abs(img.astype(float) - spiegel.astype(float)))

def symmetrie_verticaal(img):
    spiegel = img[::-1]
    return np.mean(np.abs(img.astype(float) - spiegel.astype(float)))

def standaardafwijking(img):
    return np.std(img.astype(float))

def zwaartepunt_rij(img):
    rijen = np.arange(28).reshape(28, 1)
    return np.sum(rijen * img) / (np.sum(img) + 1e-6)

def pixels_rechtsonder(img):
    return np.sum(img[14:, 14:] > DREMPEL) / (14 * 14)

def pixels_rechtsboven(img):
    return np.sum(img[:14, 14:] > DREMPEL) / (14 * 14)

def pixels_linksonder(img):
    return np.sum(img[14:, :14] > DREMPEL) / (14 * 14)

def pixels_linksboven(img):
    return np.sum(img[:14, :14] > DREMPEL) / (14 * 14)

def extract_features(img):
    return [
        procent_donkere_pixels(img),
        links_vs_rechts(img),
        boven_vs_onder(img),
        symmetrie_horizontaal(img),
        symmetrie_verticaal(img),
        standaardafwijking(img),
        zwaartepunt_rij(img),
        pixels_rechtsonder(img),
        pixels_rechtsboven(img),
        pixels_linksonder(img),
        pixels_linksboven(img),
    ]


def extract_features_batch(X_flat):
    imgs = X_flat.reshape(-1, 28, 28)
    feature_list = []

    for img in imgs:
        feats = extract_features(img)   # Deze hebben jullie eerder gemaakt! Gebruik hier je oude extract_features functie
        feature_list.append(feats)

    return np.array(feature_list)       # shape: (n_samples, n_student_features)

X_student = extract_features_batch(X)

# combineer de standaard features met jouw nieuwe features
# Dit is dus wat we hadden: (70000, 784)
# stel je hebt zelf 11 features aangemaakt
# Dan is dit wat het wordt: (70000, 784 + 11)
X_combined = np.hstack([X, X_student])
print("Combined shape:", X_combined.shape)

X_combined_train, X_combined_test, _, _ = train_test_split(X_combined, y, test_size=0.2, random_state=42)

dt_combined = DecisionTreeClassifier(criterion='gini', max_depth=20, random_state=42)
dt_combined.fit(X_combined_train, y_train)
y_pred_combined = dt_combined.predict(X_combined_test)
acc_combined = accuracy_score(y_test, y_pred_combined)
print("Accuracy gecombineerde features:", acc_combined)
print("Accuracy alleen pixels:", acc)

- Heb je nu betere resultaten?

Nauwelijks. De 784 pixels bevatten al bijna alle informatie; de 11 eigen features zijn patronen die de boom al uit de pixelwaarden zelf kan afleiden.

En wat nou als je alleen je oude features gebruikt? 

In [ ]:
X_student_train, X_student_test, _, _ = train_test_split(X_student, y, test_size=0.2, random_state=42)

dt_student = DecisionTreeClassifier(criterion='gini', max_depth=20, random_state=42)
dt_student.fit(X_student_train, y_train)
y_pred_student = dt_student.predict(X_student_test)
acc_student = accuracy_score(y_test, y_pred_student)
print("Accuracy alleen eigen features:", acc_student)
print("Accuracy alleen pixels:", acc)

Met alleen de 11 eigen features is de accuracy lager dan met 784 pixels, maar hoger dan de ~39,6% van de handgebouwde P3 boom. Sklearn vindt betere splitsingen en traint op de volledige dataset van 56.000 plaatjes in plaats van 5.000.